In [ ]:
import ROOT
import pandas as pd
import numpy as np
from io import StringIO

file = "10.17182/HEPData-ins1759506-v1-csv/Table01.csv"

def read_hepdata_sections(path):
    with open(path, "r") as f:
        lines = f.read().splitlines()

    sections = {}
    current_cent = None
    current_block = []

    for line in lines:
        if line.startswith("#: CENTRALITY [pct]"):
            # salva bloco anterior
            if current_cent is not None and current_block:
                df = pd.read_csv(StringIO("\n".join(current_block)))
                df.columns = [c.strip() for c in df.columns]
                sections[current_cent] = df
                current_block = []

            current_cent = line.split(",,,")[-1].strip()

        elif current_cent is not None:
            # ignora metadados, mas mantém cabeçalho e dados
            if not line.startswith("#:") and line.strip():
                current_block.append(line)

    # salva último bloco
    if current_cent is not None and current_block:
        df = pd.read_csv(StringIO("\n".join(current_block)))
        df.columns = [c.strip() for c in df.columns]
        sections[current_cent] = df

    return sections

sections = read_hepdata_sections(file)

print("Centralidades encontradas:")
for k in sections:
    print(k)

# escolher a centralidade desejada
cent = "0.0-5.0"
df = sections[cent]

def to_num(s):
    return pd.to_numeric(s, errors="coerce")

x = to_num(df[r"$p_{T}$ [$GeV/c$]"])
xlow = to_num(df[r"$p_{T}$ [$GeV/c$] LOW"])
xhigh = to_num(df[r"$p_{T}$ [$GeV/c$] HIGH"])
y = to_num(df[r"(1/Nev)*D2(N)/DPT/DYRAP [$(GeV/c)^{-1}$]"])

stat = to_num(df["stat. +"]).abs()
syst = to_num(df["syst. +"]).abs()
syst_unc = to_num(df["syst. uncorr. +"]).abs()

syst_corr = np.sqrt(np.clip(syst**2 - syst_unc**2, 0, None))

mask = (
    x.notna() & xlow.notna() & xhigh.notna() &
    y.notna() & stat.notna() & syst.notna() &
    syst_unc.notna() & syst_corr.notna()
)

x = x[mask].to_numpy(dtype=float)
xlow = xlow[mask].to_numpy(dtype=float)
xhigh = xhigh[mask].to_numpy(dtype=float)
y = y[mask].to_numpy(dtype=float)
stat = stat[mask].to_numpy(dtype=float)
syst_unc = syst_unc[mask].to_numpy(dtype=float)
syst_corr = syst_corr[mask].to_numpy(dtype=float)

n = len(x)

g_stat = ROOT.TGraphAsymmErrors(n)
g_unc = ROOT.TGraphAsymmErrors(n)
g_corr = ROOT.TGraphAsymmErrors(n)

for i in range(n):
    exl = x[i] - xlow[i]
    exh = xhigh[i] - x[i]

    g_stat.SetPoint(i, x[i], y[i])
    g_stat.SetPointError(i, exl, exh, stat[i], stat[i])

    g_unc.SetPoint(i, x[i], y[i])
    g_unc.SetPointError(i, exl, exh, syst_unc[i], syst_unc[i])

    g_corr.SetPoint(i, x[i], y[i])
    g_corr.SetPointError(i, exl, exh, syst_corr[i], syst_corr[i])

c = ROOT.TCanvas("c", f"Centrality {cent}%", 900, 700)
c.SetLogx()
# c.SetLogy()   # removido para deixar Y linear
c.SetTicks(1, 1)

g_unc.SetTitle("")
g_unc.SetFillColorAlpha(ROOT.kOrange + 1, 0.35)
g_unc.SetLineColor(ROOT.kOrange + 1)
g_unc.SetMarkerStyle(0)

g_corr.SetFillColorAlpha(ROOT.kRed + 1, 0.25)
g_corr.SetLineColor(ROOT.kRed + 1)
g_corr.SetMarkerStyle(0)

g_stat.SetMarkerStyle(20)
g_stat.SetMarkerSize(0.9)
g_stat.SetLineColor(ROOT.kBlack)
g_stat.SetMarkerColor(ROOT.kBlack)

g_unc.Draw("A3")
g_unc.GetXaxis().SetTitle("p_{T} [GeV/c]")
g_unc.GetYaxis().SetTitle("(1/N_{ev}) d^{2}N / dp_{T} dy")
g_unc.GetXaxis().SetMoreLogLabels()
g_unc.GetXaxis().SetNoExponent()

g_unc.GetXaxis().SetLimits(np.min(xlow) * 0.9, np.max(xhigh) * 1.1)
g_unc.SetMinimum(0.0)
g_unc.SetMaximum(np.max(y) * 1.1)

g_corr.Draw("3 SAME")
g_stat.Draw("P SAME")

leg = ROOT.TLegend(0.62, 0.70, 0.88, 0.88)
leg.SetBorderSize(0)
leg.SetFillStyle(0)
leg.AddEntry(g_stat, "stat.", "p")
leg.AddEntry(g_unc, "syst. uncorr.", "f")
leg.AddEntry(g_corr, "syst. corr.", "f")
leg.Draw()

latex = ROOT.TLatex()
latex.SetNDC()
latex.SetTextSize(0.04)
latex.DrawLatex(0.16, 0.92, f"Centrality {cent}%")

c.Draw()

Centralidades encontradas:
0.0-5.0
5.0-10.0
10.0-20.0
20.0-30.0
30.0-40.0
40.0-50.0
50.0-60.0
60.0-70.0
70.0-80.0
80.0-90.0


In [2]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize
from scipy.linalg import cho_factor, cho_solve

# --------------------------------------------------
# 1) tendência suave simples em log-log
# --------------------------------------------------
logx = np.log(x)
logy = np.log(y)

# ajuste polinomial simples para a média
deg = 4
coef = np.polyfit(logx, logy, deg=deg)
mu_logy = np.polyval(coef, logx)
mu = np.exp(mu_logy)

# resíduos
r = y - mu

# --------------------------------------------------
# 2) kernel de covariância
# --------------------------------------------------
def build_covariance(x, stat, syst_unc, syst_corr, a, b, ell, jitter=1e-12):
    dx = x[:, None] - x[None, :]
    
    # parte correlacionada
    Kcorr = (a * syst_corr)[:, None] * (a * syst_corr)[None, :]
    Kcorr = Kcorr * np.exp(-0.5 * (dx / ell)**2)

    # parte diagonal: estatístico + não correlacionado
    Kdiag = np.diag(stat**2 + (b * syst_unc)**2 + jitter)

    return Kcorr + Kdiag

# --------------------------------------------------
# 3) log-verossimilhança negativa
# --------------------------------------------------
def nll_theta(theta_log, x, r, stat, syst_unc, syst_corr):
    log_a, log_b, log_ell = theta_log
    a = np.exp(log_a)
    b = np.exp(log_b)
    ell = np.exp(log_ell)

    K = build_covariance(x, stat, syst_unc, syst_corr, a, b, ell)

    try:
        cfac = cho_factor(K, lower=True, check_finite=False)
        alpha = cho_solve(cfac, r, check_finite=False)
        logdet = 2.0 * np.sum(np.log(np.diag(cfac[0])))
        n = len(r)
        return 0.5 * (r @ alpha + logdet + n * np.log(2.0 * np.pi))
    except np.linalg.LinAlgError:
        return 1e100

# --------------------------------------------------
# 4) chute inicial e ajuste
# --------------------------------------------------
theta0 = np.log([
    1.0,   # a
    1.0,   # b
    0.5    # ell em GeV/c
])

res = minimize(
    nll_theta,
    theta0,
    args=(x, r, stat, syst_unc, syst_corr),
    method="L-BFGS-B",
    bounds=[
        (np.log(1e-2), np.log(1e2)),   # a
        (np.log(1e-2), np.log(1e2)),   # b
        (np.log(1e-3), np.log(1e1)),   # ell
    ]
)

log_a_fit, log_b_fit, log_ell_fit = res.x
a_fit = np.exp(log_a_fit)
b_fit = np.exp(log_b_fit)
ell_fit = np.exp(log_ell_fit)

print("Ajuste concluído:")
print(f"a   = {a_fit:.4f}")
print(f"b   = {b_fit:.4f}")
print(f"ell = {ell_fit:.4f} GeV/c")
print(f"NLL = {res.fun:.4f}")

# --------------------------------------------------
# 5) variâncias inferidas
# --------------------------------------------------
sigma_corr_gp = a_fit * syst_corr
sigma_unc_gp = b_fit * syst_unc
sigma_total_gp = np.sqrt(stat**2 + sigma_unc_gp**2 + sigma_corr_gp**2)

sigma_total_exp = np.sqrt(stat**2 + syst_unc**2 + syst_corr**2)

# --------------------------------------------------
# 6) tabela de comparação
# --------------------------------------------------
df_compare = pd.DataFrame({
    "pT": x,
    "y": y,
    "stat_exp": stat,
    "syst_unc_exp": syst_unc,
    "syst_corr_exp": syst_corr,
    "syst_unc_gp": sigma_unc_gp,
    "syst_corr_gp": sigma_corr_gp,
    "total_exp": sigma_total_exp,
    "total_gp": sigma_total_gp,
    "ratio_unc_gp_exp": sigma_unc_gp / syst_unc,
    "ratio_corr_gp_exp": sigma_corr_gp / syst_corr,
    "ratio_total_gp_exp": sigma_total_gp / sigma_total_exp,
})

df_compare.head(10)

Ajuste concluído:
a   = 8.2623
b   = 3.9680
ell = 1.4358 GeV/c
NLL = 94.2914


,pT,y,stat_exp,syst_unc_exp,syst_corr_exp,syst_unc_gp,syst_corr_gp,total_exp,total_gp,ratio_unc_gp_exp,ratio_corr_gp_exp,ratio_total_gp_exp
0,0.110,2049.8,10.9880,39.875,140.449223,158.224318,1160.432193,146.412896,1171.220963,3.968008,8.26229,7.999439
1,0.130,2187.3,10.0680,42.552,103.645014,168.846675,856.345145,112.491450,872.890355,3.968008,8.26229,7.759615
2,0.150,2291.6,10.0890,44.584,103.266729,176.909668,853.219647,112.931565,871.425605,3.968008,8.26229,7.716404
3,0.170,2357.6,10.1010,45.869,102.110717,182.008558,843.668337,112.394812,863.136958,3.968008,8.26229,7.679509
4,0.190,2384.7,10.0420,46.399,102.529920,184.111602,847.131913,112.987138,866.966206,3.968008,8.26229,7.673141
5,0.225,2356.9,8.8146,45.862,102.398813,181.980782,846.048675,112.545711,865.443853,3.968008,8.26229,7.689710
6,0.275,2250.6,8.3998,43.801,100.728834,173.802717,832.250822,110.160711,850.246654,3.968008,8.26229,7.718239
7,0.325,2306.0,2.8195,28.577,47.313530,113.393764,390.918101,55.345864,407.041837,3.968008,8.26229,7.354512
8,0.375,2131.4,2.3435,22.901,42.198298,90.871351,348.654568,48.069160,360.309731,3.968008,8.26229,7.495653
9,0.425,1937.2,2.2377,20.457,39.804920,81.173539,328.879787,44.809908,338.756646,3.968008,8.26229,7.559860


In [3]:
summary_table = pd.DataFrame({
    "parametro": ["a", "b", "ell [GeV/c]"],
    "valor_inferido": [a_fit, b_fit, ell_fit],
    "interpretacao": [
        "escala do sistematico correlacionado",
        "escala do sistematico nao correlacionado",
        "comprimento de correlacao em pT"
    ]
})

summary_table

,parametro,valor_inferido,interpretacao
0,a,8.262290,escala do sistematico correlacionado
1,b,3.968008,escala do sistematico nao correlacionado
2,ell [GeV/c],1.435792,comprimento de correlacao em pT
